# 🏦 FinXAI — Kredi Risk Tahmini ve Açıklanabilir Yapay Zeka (XAI)

Bu notebook, kredi risk tahmini için bir **Random Forest** modeli eğitir, **SHAP** ile açıklanabilirlik analizi yapar ve bir **Flask API** sunucusu oluşturarak frontend arayüzüne canlı tahmin hizmeti sunar.

## İçindekiler

1. **Kurulum & İmportlar**
2. **Veri Yükleme & Ön İnceleme**
3. **Model Eğitimi** (Random Forest)
4. **Model Değerlendirme** (Classification Report, ROC-AUC, Confusion Matrix)
5. **Global Özellik Önem Dereceleri** (Feature Importance)
6. **SHAP Analizi** (Summary Plot, Dependence Plot)
7. **Model & Artifact Kaydetme**
8. **Flask API Sunucusu** (SHAP Tabanlı Per-Instance Açıklama)
9. **API Test & Doğrulama**
10. **Ngrok ile Dış Erişim** (Opsiyonel)


---
## 1. Kurulum & İmportlar


In [ ]:
# Gerekli kütüphanelerin kurulumu (Colab ortamı için)
# !pip install shap flask flask-cors joblib pyngrok -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)

print("✅ Tüm kütüphaneler başarıyla yüklendi.")

---
## 2. Veri Yükleme & Ön İnceleme

> **Not:** Veri dosyaları (`X_train_preprocessed.csv`, vb.) Google Colab ortamına yüklenmiş olmalıdır.
> Colab'da dosya yükleme için `files.upload()` kullanabilirsiniz.


In [ ]:
# Google Colab kullanıyorsanız dosya yüklemek için:
# from google.colab import files
# uploaded = files.upload()

X_train = pd.read_csv("X_train_preprocessed.csv")
y_train = pd.read_csv("y_train_preprocessed.csv").squeeze()
X_test  = pd.read_csv("X_test_preprocessed.csv")
y_test  = pd.read_csv("y_test_preprocessed.csv").squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Özellikler: {X_train.columns.tolist()}")
print(f"\nHedef dağılımı (train):\n{y_train.value_counts()}")

---
## 3. Model Eğitimi (Random Forest)

`n_estimators=200` ağaç ile eğitim yapılır. Random Forest, ölçeklendirme gerektirmez.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

auc_rf = roc_auc_score(y_test, y_prob_rf)
print(f"✅ Model eğitimi tamamlandı.")
print(f"   ROC-AUC Skoru: {auc_rf:.4f}")

---
## 4. Model Değerlendirme


In [ ]:
print("=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
# ROC Eğrisi
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(7, 6))
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {auc_rf:.3f})", color="#6366f1", lw=2)
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Eğrisi")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall Eğrisi
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_prob_rf)

plt.figure(figsize=(7, 6))
plt.plot(recall_rf, precision_rf, color="#10b981", lw=2, label="Random Forest")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Eğrisi")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Farklı eşik değerlerinde performans
print("Eşik Analizi:")
print("-" * 45)
for threshold in [0.3, 0.4, 0.5]:
    y_pred_t = (y_prob_rf >= threshold).astype(int)
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f = f1_score(y_test, y_pred_t)
    print(f"Eşik={threshold:.1f}  →  Precision={p:.3f}  Recall={r:.3f}  F1={f:.3f}")

---
## 5. Global Özellik Önem Dereceleri (Feature Importance)

Random Forest'ın `feature_importances_` niteliği, eğitim sırasında her özelliğin **tüm veri seti üzerindeki** ortalama önem derecesini verir. Bu değerler **sabittir** ve yeni bir tahmin yapıldığında değişmez.

> ⚠️ **Dikkat:** Global feature importance, tek bir müşterinin tahminini açıklamak için yeterli değildir. Bunun için **SHAP** kullanılmalıdır (bir sonraki bölüm).


In [ ]:
importances = rf_model.feature_importances_

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print("Top 10 Özellik (Global):")
print(feature_importance.head(10).to_string(index=False))

# Görselleştirme
top_features = feature_importance.head(10)

plt.figure(figsize=(8, 6))
bars = plt.barh(top_features["feature"][::-1], top_features["importance"][::-1], color="#6366f1")
plt.xlabel("Önem Derecesi")
plt.title("Top 10 Özellik Önem Dereceleri (Global)")
plt.tight_layout()
plt.show()

---
## 6. SHAP Analizi (Per-Instance Açıklama)

**SHAP (SHapley Additive exPlanations)**, her bir tahmin için her özelliğin **o spesifik tahmine** ne kadar katkı yaptığını hesaplar.

- **Pozitif SHAP değeri** → Özellik, risk olasılığını **artırıyor**
- **Negatif SHAP değeri** → Özellik, risk olasılığını **azaltıyor**

Bu, global feature importance'tan farklıdır: SHAP, her müşteri için farklı değerler üretir.


In [ ]:
# SHAP Explainer (TreeExplainer, Random Forest için optimize)
X_shap = X_test.sample(500, random_state=42)

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_shap)

print(f"SHAP values tipi: {type(shap_values)}")
print(f"SHAP values shape: {np.array(shap_values).shape}")

In [ ]:
# Class 1 (Riskli) için SHAP Summary Plot
# shap_values[1] → class=1 (default) tahmini için SHAP değerleri
shap_values_class1 = shap_values[1] if isinstance(shap_values, list) else shap_values[:, :, 1]

shap.summary_plot(shap_values_class1, X_shap)

In [ ]:
# SHAP Bar Plot (Ortalama Mutlak SHAP)
shap.summary_plot(shap_values_class1, X_shap, plot_type="bar")

In [ ]:
# SHAP Dependence Plot — PAY_0
shap.dependence_plot("PAY_0", shap_values_class1, X_shap)

In [ ]:
# Tek bir müşteri örneği için SHAP açıklaması
sample_idx = 0
sample_shap = shap_values_class1[sample_idx]
sample_data = X_shap.iloc[sample_idx]

print("Tek Müşteri SHAP Açıklaması:")
print("-" * 45)
explanation = pd.DataFrame({
    "feature": X_shap.columns,
    "value": sample_data.values,
    "shap_value": sample_shap
}).sort_values(by="shap_value", key=abs, ascending=False)

print(explanation.head(10).to_string(index=False))
print(f"\nTahmin Olasılığı: {rf_model.predict_proba(sample_data.values.reshape(1, -1))[0][1]:.4f}")

---
## 7. Model & Artifact Kaydetme

Model, özellik adları ve SHAP explainer kaydedilir. Flask API bu dosyaları yükleyerek tahmin yapar.


In [ ]:
joblib.dump(rf_model, "credit_risk_model.pkl")
joblib.dump(X_train.columns.tolist(), "feature_names.pkl")

print("✅ Kaydedilen dosyalar:")
print("   - credit_risk_model.pkl (Random Forest modeli)")
print("   - feature_names.pkl (Özellik adları listesi)")
print(f"\n   Toplam özellik sayısı: {len(X_train.columns)}")
print(f"   Özellikler: {X_train.columns.tolist()}")

---
## 8. Flask API Sunucusu (SHAP Tabanlı)

Bu API, her tahmin isteği için:
1. Random Forest ile risk tahmini yapar
2. **SHAP TreeExplainer** ile per-instance özellik açıklamaları hesaplar
3. En etkili 5 özelliği, SHAP değerleriyle birlikte döndürür

### API Endpoint

| Method | Path | Açıklama |
|--------|------|----------|
| `POST` | `/predict` | Müşteri verisi alır, risk tahmini ve SHAP açıklamaları döndürür |
| `GET` | `/health` | API sağlık kontrolü |

### İstek Formatı

```json
{
  "LIMIT_BAL": 20000,
  "AGE": 24,
  "PAY_0": 2,
  "PAY_2": 2,
  ...
}
```

### Yanıt Formatı

```json
{
  "prediction": 1,
  "risk_probability": 0.85,
  "risk_label": "Riskli",
  "top_explanations": [
    {
      "feature": "PAY_0",
      "value": 2.0,
      "importance": 0.1523,
      "shap_value": 0.1523
    },
    ...
  ]
}
```

> **Not:** `importance` = `|shap_value|` (mutlak), `shap_value` = yönlü SHAP (pozitif → risk artırır, negatif → risk azaltır)


In [ ]:
%%writefile app.py
"""
FinXAI Flask API — Kredi Risk Tahmini ve SHAP Açıklamaları
==========================================================
Her tahmin isteği için SHAP TreeExplainer kullanarak per-instance
özellik önem dereceleri hesaplar.
"""

from flask import Flask, request, jsonify
from flask_cors import CORS
import pandas as pd
import numpy as np
import joblib
import shap
import logging

# ── Uygulama Yapılandırması ──────────────────────────────────────
app = Flask(__name__)
CORS(app)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Model & Explainer Yükleme ────────────────────────────────────
logger.info("Model ve explainer yükleniyor...")
model = joblib.load("credit_risk_model.pkl")
feature_names = joblib.load("feature_names.pkl")
explainer = shap.TreeExplainer(model)
logger.info(f"✅ Model yüklendi. Özellik sayısı: {len(feature_names)}")


@app.route("/health", methods=["GET"])
def health():
    """API sağlık kontrolü."""
    return jsonify({
        "status": "ok",
        "model_type": type(model).__name__,
        "feature_count": len(feature_names),
    })


@app.route("/predict", methods=["POST"])
def predict():
    """
    Kredi risk tahmini ve SHAP açıklamaları.
    
    Gelen JSON verisinden bir DataFrame oluşturur, model ile tahmin yapar
    ve SHAP değerleri ile per-instance açıklamalar döndürür.
    """
    try:
        data = request.get_json()
        if not data:
            return jsonify({"error": "Boş istek gövdesi"}), 400

        # DataFrame oluştur ve eksik sütunları 0 ile doldur
        df = pd.DataFrame([data])
        df = df.reindex(columns=feature_names, fill_value=0)

        # ── Tahmin ──
        prediction = int(model.predict(df)[0])
        probabilities = model.predict_proba(df)[0]
        risk_probability = float(probabilities[1])

        # ── SHAP Açıklamaları (Per-Instance) ──
        shap_values = explainer.shap_values(df)

        # shap_values formatı:
        # - list of 2 arrays [class_0_array, class_1_array] for RF
        # - veya 3D array (n_samples, n_features, n_classes)
        if isinstance(shap_values, list):
            # class 1 (riskli) için SHAP değerleri
            shap_vals = shap_values[1][0]
        else:
            shap_vals = shap_values[0, :, 1]

        # Açıklama listesi oluştur
        explanations = []
        for feature, value, sv in zip(feature_names, df.iloc[0].values, shap_vals):
            explanations.append({
                "feature": feature,
                "value": float(value),
                "importance": float(abs(sv)),   # Mutlak önem (sıralama için)
                "shap_value": float(sv),        # Yönlü SHAP (+ risk artırır, - azaltır)
            })

        # En etkili 5 özelliği seç (mutlak SHAP değerine göre)
        explanations = sorted(explanations, key=lambda x: x["importance"], reverse=True)[:5]

        result = {
            "prediction": prediction,
            "risk_probability": risk_probability,
            "risk_label": "Riskli" if prediction == 1 else "Risksiz",
            "top_explanations": explanations,
        }

        logger.info(
            f"Tahmin: {prediction} | Olasılık: {risk_probability:.4f} | "
            f"En etkili: {explanations[0]['feature']} (SHAP={explanations[0]['shap_value']:.4f})"
        )

        return jsonify(result)

    except Exception as e:
        logger.error(f"Tahmin hatası: {str(e)}", exc_info=True)
        return jsonify({"error": f"Tahmin sırasında hata oluştu: {str(e)}"}), 500


if __name__ == "__main__":
    print("=" * 50)
    print("FinXAI API Sunucusu Başlatılıyor...")
    print(f"Model: {type(model).__name__}")
    print(f"Özellikler: {len(feature_names)} adet")
    print("=" * 50)
    app.run(host="0.0.0.0", port=5000, debug=True)

---
## 9. API Testi & Doğrulama

Sunucuyu arka planda başlatıp bir test isteği gönderelim.


In [ ]:
# Flask sunucusunu arka planda başlat
!nohup python app.py > flask.log 2>&1 &
import time
time.sleep(3)
print("✅ Flask sunucusu arka planda başlatıldı (port 5000)")

In [ ]:
# API Test İsteği
import requests

sample = X_test.iloc[0].to_dict()
print("Gönderilen veri (ilk müşteri):")
print({k: sample[k] for k in list(sample)[:6]}, "...")

try:
    response = requests.post(
        "http://127.0.0.1:5000/predict",
        json=sample,
        timeout=10
    )
    
    if response.status_code == 200:
        result = response.json()
        print(f"\n✅ API Yanıtı:")
        print(f"   Tahmin: {result['prediction']} ({result['risk_label']})")
        print(f"   Risk Olasılığı: {result['risk_probability']:.4f}")
        print(f"\n   En Etkili Özellikler (SHAP):")
        for exp in result['top_explanations']:
            direction = "↑ Risk artırıcı" if exp['shap_value'] > 0 else "↓ Risk azaltıcı"
            print(f"   - {exp['feature']}: SHAP={exp['shap_value']:.4f} ({direction})")
    else:
        print(f"❌ Hata: {response.status_code}")
        print(response.text)
except Exception as e:
    print(f"❌ Bağlantı hatası: {e}")

In [ ]:
# SHAP Farkını Doğrula: Aynı müşteri, farklı LIMIT_BAL ile
import requests

sample_original = X_test.iloc[0].to_dict()
sample_modified = sample_original.copy()
sample_modified["LIMIT_BAL"] = 500000  # Orijinal: düşük → Yeni: yüksek limit

print("LIMIT_BAL Değişikliği Testi:")
print(f"  Orijinal LIMIT_BAL: {sample_original['LIMIT_BAL']}")
print(f"  Yeni LIMIT_BAL:     {sample_modified['LIMIT_BAL']}")
print()

try:
    r1 = requests.post("http://127.0.0.1:5000/predict", json=sample_original, timeout=10).json()
    r2 = requests.post("http://127.0.0.1:5000/predict", json=sample_modified, timeout=10).json()
    
    print(f"Orijinal Risk: {r1['risk_probability']:.4f} → Yeni Risk: {r2['risk_probability']:.4f}")
    print()
    
    # LIMIT_BAL'ın SHAP değerini karşılaştır
    lb_orig = next((e for e in r1['top_explanations'] if e['feature'] == 'LIMIT_BAL'), None)
    lb_new  = next((e for e in r2['top_explanations'] if e['feature'] == 'LIMIT_BAL'), None)
    
    print("LIMIT_BAL SHAP Karşılaştırması:")
    if lb_orig:
        print(f"  Orijinal: importance={lb_orig['importance']:.4f}, shap={lb_orig['shap_value']:.4f}")
    else:
        print(f"  Orijinal: Top 5'e girmedi")
    if lb_new:
        print(f"  Yeni:     importance={lb_new['importance']:.4f}, shap={lb_new['shap_value']:.4f}")
    else:
        print(f"  Yeni:     Top 5'e girmedi")
    
    if lb_orig and lb_new:
        if abs(lb_orig['shap_value'] - lb_new['shap_value']) > 0.001:
            print("\n✅ DOĞRULANDI: LIMIT_BAL değiştikçe SHAP değeri de değişiyor!")
        else:
            print("\n⚠️ SHAP değeri değişmedi — kontrol edin.")
    
except Exception as e:
    print(f"❌ Bağlantı hatası: {e}")

---
## 10. Ngrok ile Dış Erişim (Opsiyonel)

Frontend uygulamasının bu API'ye dışarıdan erişebilmesi için **ngrok** tüneli kullanılır.

> ⚠️ **Güvenlik:** Auth token'ınızı ortam değişkenine kaydedin, notebook'a yazmayın.
> ```bash
> export NGROK_AUTH_TOKEN="your_token_here"
> ```


In [ ]:
import os
from pyngrok import ngrok

# Token'ı ortam değişkeninden oku
token = os.environ.get("NGROK_AUTH_TOKEN")
if token:
    ngrok.set_auth_token(token)
    public_url = ngrok.connect(5000)
    print(f"✅ Ngrok tüneli açıldı:")
    print(f"   {public_url}")
    print(f"\n   Frontend .env.local'a ekleyin:")
    print(f'   NEXT_PUBLIC_API_URL={public_url}')
else:
    print("⚠️ NGROK_AUTH_TOKEN ortam değişkeni bulunamadı.")
    print("   Manuel olarak ayarlamak için:")
    print('   ngrok.set_auth_token("YOUR_TOKEN")')
    print('   public_url = ngrok.connect(5000)')

---
## 🛑 Sunucu Yönetimi

Sunucuyu durdurmak için:

```bash
# Aktif ngrok süreçlerini kontrol et
!ps aux | grep ngrok
!lsof -i :4040

# Flask sunucusunu durdur
!pkill -f "python app.py"
```
